In [ ]:
import pandas as pd
raw = "data/raw/"

orders = pd.read_csv(raw + "olist_orders_dataset.csv")
print(orders.shape)
print(orders.head())
print(orders.isna().sum())
print(orders["order_status"].value_counts())
print(orders["order_id"].is_unique)

(99441, 8)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00       

In [ ]:
customers = pd.read_csv(raw + "olist_customers_dataset.csv")
print(customers.shape)
print(customers["customer_id"].is_unique)
print(customers["customer_unique_id"].is_unique)
print(customers["customer_unique_id"].nunique())

(99441, 5)
True
False
96096


In [ ]:
import pandas as pd
raw = "data/raw/"

files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "geo": "olist_geolocation_dataset.csv",
    "trans": "product_category_name_translation.csv",
}
tables = {name: pd.read_csv(raw + f) for name, f in files.items()}

In [ ]:
for name, df in tables.items():
    print(f"=== {name} ===")
    print("rows, cols:", df.shape)
    print("fully duplicated rows:", df.duplicated().sum())
    nulls = df.isna().sum()
    print("columns with nulls:\n", nulls[nulls > 0] if nulls.any() else "none")
    print()

=== orders ===
rows, cols: (99441, 8)
fully duplicated rows: 0
columns with nulls:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

=== items ===
rows, cols: (112650, 7)
fully duplicated rows: 0
columns with nulls:
 none

=== customers ===
rows, cols: (99441, 5)
fully duplicated rows: 0
columns with nulls:
 none

=== products ===
rows, cols: (32951, 9)
fully duplicated rows: 0
columns with nulls:
 product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== sellers ===
rows, cols: (3095, 4)
fully duplicated rows: 0
columns with nulls:
 none

=== reviews ===
rows, cols: (99224, 7)
fully duplicated rows: 0
columns with nulls:
 review_comment_title      87656
review_comment_message    

In [ ]:
t = tables

# Are the supposed primary keys unique?
print("orders.order_id unique:", t["orders"]["order_id"].is_unique)
print("customers.customer_id unique:", t["customers"]["customer_id"].is_unique)
print("customers.customer_unique_id unique:", t["customers"]["customer_unique_id"].is_unique)
print("customers distinct customer_unique_id:", t["customers"]["customer_unique_id"].nunique())
print("products.product_id unique:", t["products"]["product_id"].is_unique)
print("sellers.seller_id unique:", t["sellers"]["seller_id"].is_unique)

# Order items: composite key
print("items (order_id, order_item_id) unique:",
      not t["items"].duplicated(["order_id", "order_item_id"]).any())

# Reviews: one per order?
print("reviews rows:", len(t["reviews"]), "| distinct order_id:", t["reviews"]["order_id"].nunique())

# Foreign keys: do all items point to a real order / product / seller?
print("items with unknown order:", (~t["items"]["order_id"].isin(t["orders"]["order_id"])).sum())
print("items with unknown product:", (~t["items"]["product_id"].isin(t["products"]["product_id"])).sum())
print("items with unknown seller:", (~t["items"]["seller_id"].isin(t["sellers"]["seller_id"])).sum())

# Orders without any item?
print("orders with no items:", (~t["orders"]["order_id"].isin(t["items"]["order_id"])).sum())


orders.order_id unique: True
customers.customer_id unique: True
customers.customer_unique_id unique: False
customers distinct customer_unique_id: 96096
products.product_id unique: True
sellers.seller_id unique: True
items (order_id, order_item_id) unique: True
reviews rows: 99224 | distinct order_id: 98673
items with unknown order: 0
items with unknown product: 0
items with unknown seller: 0
orders with no items: 775


In [ ]:
print(t["orders"]["order_status"].value_counts(), "\n")

dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in dates:
    s = pd.to_datetime(t["orders"][c])
    print(c, "->", s.min(), "to", s.max())

print("\nprice:\n", t["items"]["price"].describe())
print("\nfreight_value:\n", t["items"]["freight_value"].describe())
print("\nreview_score counts:\n", t["reviews"]["review_score"].value_counts().sort_index())

# Delivered orders with no delivery date, and delivery before purchase
o = t["orders"].copy()
for c in ["order_purchase_timestamp", "order_delivered_customer_date"]:
    o[c] = pd.to_datetime(o[c])
print("\ndelivered but no delivery date:",
      ((o["order_status"] == "delivered") & o["order_delivered_customer_date"].isna()).sum())
print("delivered before purchased:",
      (o["order_delivered_customer_date"] < o["order_purchase_timestamp"]).sum())

# Products without an English category translation
prod = t["products"].merge(t["trans"], on="product_category_name", how="left")
print("products with no category:", prod["product_category_name"].isna().sum())
print("products with category but no translation:",
      (prod["product_category_name"].notna() & prod["product_category_name_english"].isna()).sum())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64 

order_purchase_timestamp -> 2016-09-04 21:15:19 to 2018-10-17 17:30:18
order_delivered_customer_date -> 2016-10-11 13:46:32 to 2018-10-17 13:22:46
order_estimated_delivery_date -> 2016-09-30 00:00:00 to 2018-11-12 00:00:00

price:
 count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

freight_value:
 count    112650.000000
mean         19.990320
std          15.806405
min           0.000000
25%          13.080000
50%          16.260000
75%          21.150000
max         409.680000
Name: freight_value, dtype: float64

review_score counts:
 review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype:

In [ ]:
t = tables
print("review_id rows:", len(t["reviews"]), "| distinct review_id:", t["reviews"]["review_id"].nunique())
print("orders with >1 review:", (t["reviews"].groupby("order_id").size() > 1).sum())
print("payments per order > 1:", (t["payments"].groupby("order_id").size() > 1).sum())
print("payment_value == 0:", (t["payments"]["payment_value"] == 0).sum())
print(t["payments"]["payment_type"].value_counts())
print("geo rows:", len(t["geo"]), "| duplicate rows:", t["geo"].duplicated().sum(),
      "| distinct zip prefixes:", t["geo"]["geolocation_zip_code_prefix"].nunique())
print("freight_value == 0:", (t["items"]["freight_value"] == 0).sum())
print("distinct orders in items:", t["items"]["order_id"].nunique())
print(t["customers"]["customer_state"].nunique(), "states")

review_id rows: 99224 | distinct review_id: 98410
orders with >1 review: 547
payments per order > 1: 2961
payment_value == 0: 9
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
geo rows: 1000163 | duplicate rows: 261831 | distinct zip prefixes: 19015
freight_value == 0: 383
distinct orders in items: 98666
27 states


In [ ]:
t = tables
o = t["orders"].copy()
for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    o[c] = pd.to_datetime(o[c])

print("orders rows:", len(o))
print("items rows:", len(t["items"]))
print("customers rows:", len(t["customers"]), "| distinct customer_unique_id:", t["customers"]["customer_unique_id"].nunique())
print("null approved_at:", o["order_approved_at"].isna().sum())
print("null delivered_carrier:", o["order_delivered_carrier_date"].isna().sum())
print("null delivered_customer:", o["order_delivered_customer_date"].isna().sum())
print("delivered but no delivery date:", ((o["order_status"]=="delivered") & o["order_delivered_customer_date"].isna()).sum())
print("delivered before purchase:", (o["order_delivered_customer_date"] < o["order_purchase_timestamp"]).sum())
print("orders with no items:", (~o["order_id"].isin(t["items"]["order_id"])).sum())
print(o["order_status"].value_counts())
print("purchase range:", o["order_purchase_timestamp"].min(), "to", o["order_purchase_timestamp"].max())
print(o["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index())
prod = t["products"].merge(t["trans"], on="product_category_name", how="left")
print("products rows:", len(prod))
print("products with no category:", prod["product_category_name"].isna().sum())
print("categories with no translation:", prod.loc[prod["product_category_name"].notna() & prod["product_category_name_english"].isna(), "product_category_name"].unique())
print("review_score values:", sorted(t["reviews"]["review_score"].unique()))
print("min price:", t["items"]["price"].min(), "| max price:", t["items"]["price"].max())

orders rows: 99441
items rows: 112650
customers rows: 99441 | distinct customer_unique_id: 96096
null approved_at: 160
null delivered_carrier: 1783
null delivered_customer: 2965
delivered but no delivery date: 8
delivered before purchase: 0
orders with no items: 775
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
purchase range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64
product